In [1]:
import torch

print("Checking GPU...")
print("="*50)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Found:", torch.cuda.get_device_name(0))
    print("GPU Memory:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")
else:
    print("NO GPU! Select GPU from Accelerator")

Checking GPU...
CUDA available: True
GPU Found: Tesla T4
GPU Memory: 15.637086208 GB


In [2]:
print("Installing packages...")
!pip install -q transformers datasets accelerate scikit-learn

print("Done!")
import transformers
print(f"transformers: {transformers.__version__}")

Installing packages...
Done!
transformers: 4.57.1


In [3]:
import os

print("Searching for files...")
print("="*50)

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith('.csv'):
            full_path = os.path.join(root, file)
            size = os.path.getsize(full_path)
            print(f"Found: {file}")
            print(f"  Path: {full_path}")
            print(f"  Size: {size:,} bytes\n")

Searching for files...
Found: test_with_labels.csv
  Path: /kaggle/input/datasets/rishta123/testdataset/test_with_labels.csv
  Size: 265,764 bytes

Found: TestV2 - testV2.csv
  Path: /kaggle/input/tamil-text/TestV2 - testV2.csv
  Size: 256,503 bytes

Found: trainV2.csv
  Path: /kaggle/input/tamil-text/trainV2.csv
  Size: 1,109,007 bytes



In [4]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

train_check = pd.read_csv("/kaggle/input/tamil-text/trainV2.csv")
print(train_check.head(3))
print(f"\nColumns: {train_check.columns.tolist()}")
print(f"Train size: {len(train_check)}")

                                                Text        Class
0  நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சி...  Non-Abusive
1  உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங...      Abusive
2  கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்...  Non-Abusive

Columns: ['Text', 'Class']
Train size: 3652


In [5]:
true_labels_train = train_check["Class"].values
print(f"Label distribution:\n{train_check['Class'].value_counts()}")

Label distribution:
Class
Non-Abusive    1883
Abusive        1768
abusive           1
Name: count, dtype: int64


In [6]:
# ── Normalize labels to binary classes ───────────────────────────────────────
train_check["Class"] = train_check["Class"].str.strip().str.lower().map({
    "abusive":     "Abusive",
    "non-abusive": "Non-Abusive"
})

# Verify
print(train_check["Class"].value_counts())
print(f"Nulls after mapping: {train_check['Class'].isna().sum()}")

true_labels_train = train_check["Class"].values

Class
Non-Abusive    1883
Abusive        1769
Name: count, dtype: int64
Nulls after mapping: 0


# Preprocessing

In [7]:
import shutil
import pandas as pd
import re
import html as _html
import numpy as np

TRAIN_PATH = "/kaggle/input/tamil-text/trainV2.csv"
TEST_PATH = "/kaggle/input/tamil-text/TestV2 - testV2.csv"

shutil.copy(TRAIN_PATH, "trainV2.csv")
shutil.copy(TEST_PATH, "TestV2 - testV2.csv")
print("✓ Files copied!")

# Advanced text cleaning function
def clean_text(text):
    """Comprehensive text cleaning for Tamil abuse detection"""
    
    if pd.isna(text) or text == '':
        return ""
    
    text = str(text)
    
    # 1. HTML entities (e.g., &amp; → &)
    text = _html.unescape(text)
    
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # 3. Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # 4. Remove mentions (@username)
    text = re.sub(r'@\w+', '', text)
    
    # 5. Remove hashtags (keep the text, remove #)
    text = re.sub(r'#(\w+)', r'\1', text)
    
    # 6. Remove excessive punctuation (more than 2 in a row)
    text = re.sub(r'([!?.,]){3,}', r'\1\1', text)
    
    # 7. Remove extra spaces, newlines, tabs
    text = re.sub(r'\s+', ' ', text)
    
    # 8. Remove leading/trailing whitespace
    text = text.strip()
    

    # 9. Remove standalone numbers (keep numbers in words)
    text = re.sub(r'\b\d+\b', '', text)
    
    # 10. Normalize quotes
    text = text.replace('"', '"').replace('"', '"')
    text = text.replace("'", "'").replace("'", "'")
    
    # 11. Remove zero-width characters (invisible Unicode)
    text = text.replace('\u200b', '').replace('\u200c', '').replace('\u200d', '')
    
    return text

# Process training data

print("\n" + "="*50)
print("PROCESSING TRAINING DATA")
print("="*50)

df = pd.read_csv("trainV2.csv")
print(f"Original rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

# Check data before cleaning
print(f"\nBefore cleaning:")
print(f"  Null texts: {df['Text'].isna().sum()}")
print(f"  Empty texts: {(df['Text'] == '').sum()}")

# Apply text cleaning
print("\nApplying preprocessing...")
df["Text"] = df["Text"].apply(clean_text)

# Standardize labels
df["Class"] = df["Class"].astype(str).str.strip()
df["Class"] = df["Class"].replace({
    "abusive": "Abusive",
    "ABUSIVE": "Abusive",
    "non-abusive": "Non-Abusive",
    "NON-ABUSIVE": "Non-Abusive",
    "Non abusive": "Non-Abusive",
    "NonAbusive": "Non-Abusive",
    "non abusive": "Non-Abusive"
})

# Remove rows with empty text after cleaning
before_removal = len(df)
df = df[df['Text'].str.len() > 0].copy()
after_removal = len(df)
if before_removal != after_removal:
    print(f"  Removed {before_removal - after_removal} empty texts after cleaning")


# Remove duplicates
before_dedup = len(df)
df = df.drop_duplicates(subset=['Text'], keep='first').reset_index(drop=True)
after_dedup = len(df)
if before_dedup != after_dedup:
    print(f"  Removed {before_dedup - after_dedup} duplicate texts")

print(f"\nFinal rows: {len(df)}")
print("\nLabel distribution:")
print(df["Class"].value_counts())
print("\nLabel percentages:")
print(df["Class"].value_counts(normalize=True) * 100)

# Save cleaned data
df.to_csv("train_clean.csv", index=False)
print("\n✓ Saved: train_clean.csv")

# Process test data (same cleaning, no labels)

print("\n" + "="*50)
print("PROCESSING TEST DATA")
print("="*50)

test_df = pd.read_csv("TestV2 - testV2.csv")
print(f"Test rows: {len(test_df)}")

# Apply same text cleaning
test_df["Text"] = test_df["Text"].apply(clean_text)

# Remove empty texts
before_test = len(test_df)
test_df = test_df[test_df['Text'].str.len() > 0].copy()
after_test = len(test_df)
if before_test != after_test:
    print(f"  Removed {before_test - after_test} empty texts")

print(f"Final test rows: {len(test_df)}")

# Save cleaned test
test_df.to_csv("TestV2_clean.csv", index=False)
print("✓ Saved: TestV2_clean.csv")

# Show preprocessing examples
print("\n" + "="*50)
print("PREPROCESSING EXAMPLES")
print("="*50)

# Load original to compare
df_original = pd.read_csv("trainV2.csv")

sample_indices = [0, 1, 2]
for i in sample_indices:
    if i < len(df_original):
        original = str(df_original['Text'].iloc[i])
        cleaned = str(df['Text'].iloc[i])
        
        if original != cleaned:
            print(f"\nExample {i+1}:")
            print(f"BEFORE: {original[:150]}")
            print(f"AFTER:  {cleaned[:150]}")
            print(f"Label:  {df['Class'].iloc[i]}")

# Text statistics

print("\n" + "="*50)
print("TEXT STATISTICS")
print("="*50)

lengths = df['Text'].str.len()
word_counts = df['Text'].str.split().str.len()

print(f"\nCharacter lengths:")
print(f"  Min: {lengths.min()}")
print(f"  Max: {lengths.max()}")
print(f"  Mean: {lengths.mean():.1f}")
print(f"  Median: {lengths.median():.1f}")

print(f"\nWord counts:")
print(f"  Min: {word_counts.min()}")
print(f"  Max: {word_counts.max()}")
print(f"  Mean: {word_counts.mean():.1f}")
print(f"  Median: {word_counts.median():.1f}")

# NEW: Show distribution percentiles
print(f"\nCharacter length percentiles:")
print(f"  25th: {lengths.quantile(0.25):.0f}")
print(f"  50th (median): {lengths.quantile(0.50):.0f}")
print(f"  75th: {lengths.quantile(0.75):.0f}")
print(f"  95th: {lengths.quantile(0.95):.0f}")

print("\n" + "="*50)
print("✅ PREPROCESSING COMPLETE!")
print("="*50)
print("\nUse these files for training:")
print("  --train 'train_clean.csv'")
print("  --test 'TestV2_clean.csv'")

✓ Files copied!

PROCESSING TRAINING DATA
Original rows: 3652
Columns: ['Text', 'Class']

Before cleaning:
  Null texts: 0
  Empty texts: 0

Applying preprocessing...
  Removed 401 duplicate texts

Final rows: 3251

Label distribution:
Class
Non-Abusive    1667
Abusive        1584
Name: count, dtype: int64

Label percentages:
Class
Non-Abusive    51.27653
Abusive        48.72347
Name: proportion, dtype: float64

✓ Saved: train_clean.csv

PROCESSING TEST DATA
Test rows: 913
Final test rows: 913
✓ Saved: TestV2_clean.csv

PREPROCESSING EXAMPLES

Example 1:
BEFORE: நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சிட்டேன் திவ்யா. நீ தெளிவாதான் இருக்க. உங்க வீடியோ பாக்குரேன்ல நான்தான் பைத்தியமா இருந்துருக்கேன்
AFTER:  நான் கூட உன்னை வெகுளியான பொண்ணு'னு நெனச்சிட்டேன் திவ்யா. நீ தெளிவாதான் இருக்க. உங்க வீடியோ பாக்குரேன்ல நான்தான் பைத்தியமா இருந்துருக்கேன்
Label:  Non-Abusive

TEXT STATISTICS

Character lengths:
  Min: 17
  Max: 3131
  Mean: 109.6
  Median: 80.0

Word counts:
  Min: 3
  Max: 381
 

In [8]:
import shutil
import os

print("Cleaning old files...")
print("="*50)

# Delete old checkpoint folders
for item in os.listdir("."):
    if item.startswith("outputs_"):
        try:
            size = sum(
                os.path.getsize(os.path.join(dp, f))
                for dp, dn, fns in os.walk(item)
                for f in fns
            )
            shutil.rmtree(item)
            print(f"Deleted: {item} ({size/1e6:.1f} MB freed)")
        except:
            pass

# Delete old zip files
for item in os.listdir("."):
    if item.endswith(".zip"):
        os.remove(item)
        print(f"Deleted: {item}")

# Delete old submissions
if os.path.exists("submissions"):
    shutil.rmtree("submissions")
    print("Deleted: old submissions folder")

os.makedirs("submissions", exist_ok=True)
print("\nDisk cleanup done!")

Cleaning old files...

Disk cleanup done!


# Script file

In [9]:
# create the script file:

script = '''import argparse, os, html as _html, shutil, sys, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, 
    set_seed
)
import warnings
warnings.filterwarnings('ignore')

def clean_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = _html.unescape(s)
    s = " ".join(s.split())
    return s

def macro_prf1(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {"macro_precision": p, "macro_recall": r, "macro_f1": f1}

@dataclass
class SimpleDataset(torch.utils.data.Dataset):
    encodings: dict
    labels: np.ndarray | None = None

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels[idx]))
        return item

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.get("logits")

        if hasattr(model, "module"):
            num_labels = model.module.config.num_labels
        else:
            num_labels = model.config.num_labels

        if self.class_weights is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", required=True)
    ap.add_argument("--test", required=True)
    ap.add_argument("--team", required=True)
    ap.add_argument("--text_col", default="Text")
    ap.add_argument("--label_col", default="Class")
    ap.add_argument("--model", default="xlm-roberta-base")
    ap.add_argument("--max_len", type=int, default=128)
    ap.add_argument("--epochs", type=int, default=5)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--lr", type=float, default=2e-5)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--val_size", type=float, default=0.1)
    ap.add_argument("--gradient_accumulation_steps", type=int, default=1)
    ap.add_argument("--weight_decay", type=float, default=0.01)
    ap.add_argument("--warmup_ratio", type=float, default=0.1)
    ap.add_argument("--early_stopping_patience", type=int, default=3)
    ap.add_argument("--quiet", action="store_true", default=False)
    args = ap.parse_args()

    set_seed(args.seed)

    print(f"\\n{'='*70}")
    print(f"Training {args.model} (Seed {args.seed})")
    print(f"{'='*70}\\n")
    
    train_df = pd.read_csv(args.train)
    test_df = pd.read_csv(args.test)

    train_df[args.text_col] = train_df[args.text_col].map(clean_text)
    test_df[args.text_col] = test_df[args.text_col].map(clean_text)

    if train_df[args.label_col].dtype == 'object':
        label_map = {"Non-Abusive": 0, "Abusive": 1}
        y_str = train_df[args.label_col].astype(str).str.strip()
        y = y_str.map(label_map).to_numpy()
    else:
        y = train_df[args.label_col].to_numpy()
    
    inv = {0: "Non-Abusive", 1: "Abusive"}
    X = train_df[args.text_col].astype(str).to_numpy()

    X_tr, X_va, y_tr, y_va = train_test_split(
        X, y, test_size=args.val_size, random_state=args.seed, stratify=y
    )

    print(f"Train: {len(X_tr)}, Val: {len(X_va)}, Test: {len(test_df)}\\n")

    tokenizer = AutoTokenizer.from_pretrained(args.model, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(args.model, num_labels=2)

    tr_enc = tokenizer(list(X_tr), truncation=True, max_length=args.max_len)
    va_enc = tokenizer(list(X_va), truncation=True, max_length=args.max_len)
    te_enc = tokenizer(list(test_df[args.text_col].astype(str).to_numpy()), truncation=True, max_length=args.max_len)

    tr_ds = SimpleDataset(tr_enc, y_tr)
    va_ds = SimpleDataset(va_enc, y_va)
    te_ds = SimpleDataset(te_enc, labels=None)

    counts = np.bincount(y_tr, minlength=2)
    weights = (counts.sum() / (2.0 * np.maximum(counts, 1))).astype(np.float32)
    class_weights = torch.tensor(weights, dtype=torch.float32)

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return macro_prf1(labels, preds)

    output_dir = f"outputs_{args.model.replace('/', '_')}"

    targs = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="no",
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch,
        per_device_eval_batch_size=args.batch,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        learning_rate=args.lr,
        lr_scheduler_type="cosine",
        warmup_ratio=args.warmup_ratio,
        weight_decay=args.weight_decay,
        max_grad_norm=1.0,
        fp16=torch.cuda.is_available(),
        report_to="none",
        logging_steps=50,
        disable_tqdm=args.quiet,
        logging_strategy="steps",
        seed=args.seed,
        data_seed=args.seed,
    )

    trainer = WeightedTrainer(
        model=model,
        args=targs,
        train_dataset=tr_ds,
        eval_dataset=va_ds,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )

    print("Training...\\n")
    trainer.train()
    
    eval_result = trainer.evaluate()
    
    print("\\n" + "="*70)
    print("FINAL RESULTS:")
    print("="*70)
    print(f"F1: {eval_result['eval_macro_f1']:.4f} ({eval_result['eval_macro_f1']*100:.2f}%)")
    print(f"Precision: {eval_result['eval_macro_precision']:.4f}")
    print(f"Recall: {eval_result['eval_macro_recall']:.4f}")
    print(f"Loss: {eval_result['eval_loss']:.4f}")
    print("="*70 + "\\n")

    pred = trainer.predict(te_ds)
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred_labels = [inv[int(i)] for i in pred_ids]
    probs = torch.softmax(torch.tensor(pred.predictions), dim=-1).numpy()

    os.makedirs("submissions", exist_ok=True)

    csv_path = f"submissions/{args.team}.csv"
    prob_path = f"submissions/{args.team}_probs.npy"

    if "Text" in test_df.columns:
        sub = pd.DataFrame({"Text": test_df["Text"], "Class": pred_labels})
    else:
        sub = pd.DataFrame({"Class": pred_labels})

    sub.to_csv(csv_path, index=False, encoding="utf-8")
    np.save(prob_path, probs)
    print(f"✅ Created: {csv_path}")
    print(f"✅ Created: {prob_path}\\n")

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    
    print("✓ DONE!\\n")

if __name__ == "__main__":
    main()
'''

with open("train_updated.py", "w") as f:
    f.write(script)

print("✅ Created: train_updated.py")

✅ Created: train_updated.py


In [10]:
import os

print("Checking files...")
print("="*50)

required = ["train_clean.csv", "TestV2 - testV2.csv", "train_updated.py",]

all_good = True
for f in required:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f"OK: {f} ({size:,} bytes)")
    else:
        print(f"MISSING: {f}")
        all_good = False

print("="*50)
if all_good:
    print("ALL FILES READY!")
else:
    print("FIX MISSING FILES FIRST!")

Checking files...
OK: train_clean.csv (982,448 bytes)
OK: TestV2 - testV2.csv (256,503 bytes)
OK: train_updated.py (7,447 bytes)
ALL FILES READY!


In [11]:
import shutil
import os

print("Cleaning submissions folder...")

if os.path.exists("submissions"):
    shutil.rmtree("submissions")
    
os.makedirs("submissions", exist_ok=True)

print("✓ Fresh start!")

Cleaning submissions folder...
✓ Fresh start!


In [12]:
# check column names
import pandas as pd

train_df = pd.read_csv("train_clean.csv")
test_df = pd.read_csv("TestV2_clean.csv")

print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())
print("\nFirst few rows of train:")
print(train_df.head())

Train columns: ['Text', 'Class']
Test columns: ['Text']

First few rows of train:
                                                Text        Class
0  நான் கூட உன்னை வெகுளியான பொண்ணு'னு நெனச்சிட்டே...  Non-Abusive
1  உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங...      Abusive
2  கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்...  Non-Abusive
3  ஒன்னோட சைசுக்கு நீயே ஒரு நாளக்கி 5கிலோ ஆய் போவ...      Abusive
4  ரெண்டும் மிக பெரிய வெடிகுண்டு இவங்கள எதுக்கு ஷ...  Non-Abusive


# Train MuRIL

In [13]:
import shutil
import os

TEAM = "CUET_Synthetica"
SEEDS = [3407]

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Training MuRIL - Seed {seed}_1")
    print(f"{'='*60}")
    
    !python train_updated.py \
      --train "train_clean.csv" \
      --test "TestV2_clean.csv" \
      --team "$TEAM" \
      --model "google/muril-base-cased" \
      --text_col "Text" \
      --label_col "Class" \
      --epochs 10 \
      --max_len 128 \
      --lr 2e-5 \
      --batch 32 \
      --val_size 0.2 \
      --seed {seed} \
      --early_stopping_patience 3
    
    if os.path.exists("submissions/CUET_Synthetica.csv"):
        shutil.move("submissions/CUET_Synthetica.csv", 
                    f"submissions/CUET_Synthetica_MuRIL_{seed}_1.csv")
        shutil.move("submissions/CUET_Synthetica_probs.npy", 
                    f"submissions/CUET_Synthetica_MuRIL_{seed}_1_probs.npy")
        print(f"\n✅ Seed {seed} DONE\n")
    else:
        print(f"\n❌ Seed {seed} FAILED\n")

print("\n" + "="*60)
print("🎉 ALL MuRIL MODELS COMPLETE!")
print("="*60)


Training MuRIL - Seed 3407_1
2026-02-28 08:50:21.136974: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772268621.353211     126 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772268621.408670     126 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772268621.876962     126 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772268621.877012     126 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772268621.877016     126 computation_placer.

In [14]:
import shutil
import os

TEAM = "CUET_Synthetica"
SEEDS = [3407]

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Training MuRIL - Seed {seed}_2")
    print(f"{'='*60}")
    
    !python train_updated.py \
      --train "train_clean.csv" \
      --test "TestV2_clean.csv" \
      --team "$TEAM" \
      --model "google/muril-base-cased" \
      --text_col "Text" \
      --label_col "Class" \
      --epochs 10 \
      --max_len 128 \
      --lr 3e-5 \
      --batch 32 \
      --val_size 0.2 \
      --seed {seed} \
      --early_stopping_patience 3
    
    if os.path.exists("submissions/CUET_Synthetica.csv"):
        shutil.move("submissions/CUET_Synthetica.csv", 
                    f"submissions/CUET_Synthetica_MuRIL_{seed}_2.csv")
        shutil.move("submissions/CUET_Synthetica_probs.npy", 
                    f"submissions/CUET_Synthetica_MuRIL_{seed}_2_probs.npy")
        print(f"\n✅ Seed {seed} DONE\n")
    else:
        print(f"\n❌ Seed {seed} FAILED\n")

print("\n" + "="*60)
print("🎉 ALL MuRIL MODELS COMPLETE!")
print("="*60)


Training MuRIL - Seed 3407_2
2026-02-28 08:57:14.087684: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772269034.109211    1319 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772269034.115634    1319 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772269034.133219    1319 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772269034.133255    1319 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772269034.133260    1319 computation_placer.

In [15]:
import shutil
import os

TEAM = "CUET_Synthetica"
SEEDS = [3407]

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Training MuRIL - Seed {seed}")
    print(f"{'='*60}")
    
    !python train_updated.py \
      --train "train_clean.csv" \
      --test "TestV2_clean.csv" \
      --team "$TEAM" \
      --model "google/muril-base-cased" \
      --text_col "Text" \
      --label_col "Class" \
      --epochs 10 \
      --max_len 128 \
      --lr 4e-5 \
      --batch 32 \
      --val_size 0.2 \
      --seed {seed} \
      --early_stopping_patience 3
    
    if os.path.exists("submissions/CUET_Synthetica.csv"):
        shutil.move("submissions/CUET_Synthetica.csv", 
                    f"submissions/CUET_Synthetica_MuRIL_{seed}_3_.csv")
        shutil.move("submissions/CUET_Synthetica_probs.npy", 
                    f"submissions/CUET_Synthetica_MuRIL_{seed}_3_probs.npy")
        print(f"\n✅ Seed {seed} DONE\n")
    else:
        print(f"\n❌ Seed {seed} FAILED\n")

print("\n" + "="*60)
print("🎉 ALL MuRIL MODELS COMPLETE!")
print("="*60)


Training MuRIL - Seed 3407
2026-02-28 09:03:49.405436: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772269429.428199    2454 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772269429.434678    2454 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772269429.451889    2454 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772269429.451918    2454 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772269429.451924    2454 computation_placer.cc

In [16]:
import shutil
import os

TEAM = "CUET_Synthetica"
SEEDS = [3407]

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Training MuRIL - Seed {seed}_4")
    print(f"{'='*60}")
    
    !python train_updated.py \
      --train "train_clean.csv" \
      --test "TestV2_clean.csv" \
      --team "$TEAM" \
      --model "google/muril-base-cased" \
      --text_col "Text" \
      --label_col "Class" \
      --epochs 10 \
      --max_len 128 \
      --lr 5e-5 \
      --batch 32 \
      --val_size 0.2 \
      --seed {seed} \
      --early_stopping_patience 3
    
    if os.path.exists("submissions/CUET_Synthetica.csv"):
        shutil.move("submissions/CUET_Synthetica.csv", 
                    f"submissions/CUET_Synthetica_MuRIL_{seed}_4.csv")
        shutil.move("submissions/CUET_Synthetica_probs.npy", 
                    f"submissions/CUET_Synthetica_MuRIL_{seed}_4_probs.npy")
        print(f"\n✅ Seed {seed} DONE\n")
    else:
        print(f"\n❌ Seed {seed} FAILED\n")

print("\n" + "="*60)
print("🎉 ALL MuRIL MODELS COMPLETE!")
print("="*60)


Training MuRIL - Seed 3407_4
2026-02-28 09:10:24.342858: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772269824.364050    3589 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772269824.370565    3589 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772269824.388224    3589 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772269824.388251    3589 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772269824.388256    3589 computation_placer.

In [17]:
#muril done

# Train XLM-RoBERTa-Large

In [18]:
import shutil
import os

TEAM = "CUET_Synthetica"


SEEDS = [3407]

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Training XLM-RoBERTa-Large - Seed {seed}_1")
    print(f"{'='*60}")
    
    !python train_updated.py \
      --train "train_clean.csv" \
      --test "TestV2_clean.csv" \
      --team "$TEAM" \
      --model "xlm-roberta-large" \
      --text_col "Text" \
      --label_col "Class" \
      --epochs 10 \
      --max_len 128 \
      --lr 2e-6 \
      --batch 8 \
      --val_size 0.2 \
      --seed {seed} \
      --early_stopping_patience 3
    
    if os.path.exists("submissions/CUET_Synthetica.csv"):
        shutil.move("submissions/CUET_Synthetica.csv", 
                    f"submissions/CUET_Synthetica_XLMRobertaLarge_{seed}_1.csv")
        shutil.move("submissions/CUET_Synthetica_probs.npy", 
                    f"submissions/CUET_Synthetica_XLMRobertaLarge_{seed}_1_probs.npy")
        print(f"\n✅ Seed {seed} DONE\n")
    else:
        print(f"\n❌ Seed {seed} FAILED\n")

print("\n" + "="*60)
print("🎉 ALL XLM-RoBERTa-Large MODELS COMPLETE!")
print("="*60)


Training XLM-RoBERTa-Large - Seed 3407_1
2026-02-28 09:23:46.209894: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772270626.231522    4725 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772270626.238153    4725 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772270626.256117    4725 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772270626.256163    4725 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772270626.256169    4725 computa

In [19]:
import shutil
import os

TEAM = "CUET_Synthetica"

SEEDS = [3407]

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Training XLM-RoBERTa-Large - Seed {seed}_2")
    print(f"{'='*60}")
    
    !python train_updated.py \
      --train "train_clean.csv" \
      --test "TestV2_clean.csv" \
      --team "$TEAM" \
      --model "xlm-roberta-large" \
      --text_col "Text" \
      --label_col "Class" \
      --epochs 10 \
      --max_len 128 \
      --lr 3e-6 \
      --batch 8 \
      --val_size 0.2 \
      --seed {seed} \
      --early_stopping_patience 3
    
    if os.path.exists("submissions/CUET_Synthetica.csv"):
        shutil.move("submissions/CUET_Synthetica.csv", 
                    f"submissions/CUET_Synthetica_XLMRobertaLarge_{seed}_2.csv")
        shutil.move("submissions/CUET_Synthetica_probs.npy", 
                    f"submissions/CUET_Synthetica_XLMRobertaLarge_{seed}_2_probs.npy")
        print(f"\n✅ Seed {seed} DONE\n")
    else:
        print(f"\n❌ Seed {seed} FAILED\n")

print("\n" + "="*60)
print("🎉 ALL XLM-RoBERTa-Large MODELS COMPLETE!")
print("="*60)


Training XLM-RoBERTa-Large - Seed 3407_2
2026-02-28 09:54:20.615587: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772272460.637406    9121 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772272460.644119    9121 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772272460.661379    9121 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772272460.661410    9121 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772272460.661414    9121 computa

In [20]:
import shutil
import os

TEAM = "CUET_Synthetica"


SEEDS = [3407]

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Training XLM-RoBERTa-Large - Seed {seed}_3")
    print(f"{'='*60}")
    
    !python train_updated.py \
      --train "train_clean.csv" \
      --test "TestV2_clean.csv" \
      --team "$TEAM" \
      --model "xlm-roberta-large" \
      --text_col "Text" \
      --label_col "Class" \
      --epochs 10 \
      --max_len 128 \
      --lr 4e-6 \
      --batch 8 \
      --val_size 0.2 \
      --seed {seed} \
      --early_stopping_patience 3
    
    if os.path.exists("submissions/CUET_Synthetica.csv"):
        shutil.move("submissions/CUET_Synthetica.csv", 
                    f"submissions/CUET_Synthetica_XLMRobertaLarge_{seed}_3.csv")
        shutil.move("submissions/CUET_Synthetica_probs.npy", 
                    f"submissions/CUET_Synthetica_XLMRobertaLarge_{seed}_3_probs.npy")
        print(f"\n✅ Seed {seed} DONE\n")
    else:
        print(f"\n❌ Seed {seed} FAILED\n")

print("\n" + "="*60)
print("🎉 ALL XLM-RoBERTa-Large MODELS COMPLETE!")
print("="*60)


Training XLM-RoBERTa-Large - Seed 3407_3
2026-02-28 10:24:59.431967: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772274299.453532   13466 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772274299.459889   13466 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772274299.477389   13466 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772274299.477419   13466 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772274299.477423   13466 computa

In [21]:
import shutil
import os

TEAM = "CUET_Synthetica"

SEEDS = [3407]

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Training XLM-RoBERTa-Large - Seed {seed}_4")
    print(f"{'='*60}")
    
    !python train_updated.py \
      --train "train_clean.csv" \
      --test "TestV2_clean.csv" \
      --team "$TEAM" \
      --model "xlm-roberta-large" \
      --text_col "Text" \
      --label_col "Class" \
      --epochs 10 \
      --max_len 128 \
      --lr 5e-6 \
      --batch 8 \
      --val_size 0.2 \
      --seed {seed} \
      --early_stopping_patience 3
    
    if os.path.exists("submissions/CUET_Synthetica.csv"):
        shutil.move("submissions/CUET_Synthetica.csv", 
                    f"submissions/CUET_Synthetica_XLMRobertaLarge_{seed}_4.csv")
        shutil.move("submissions/CUET_Synthetica_probs.npy", 
                    f"submissions/CUET_Synthetica_XLMRobertaLarge_{seed}_4_probs.npy")
        print(f"\n✅ Seed {seed} DONE\n")
    else:
        print(f"\n❌ Seed {seed} FAILED\n")

print("\n" + "="*60)
print("🎉 ALL XLM-RoBERTa-Large MODELS COMPLETE!")
print("="*60)


Training XLM-RoBERTa-Large - Seed 3407_4
2026-02-28 10:55:40.614836: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772276140.638320   17836 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772276140.645015   17836 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772276140.661835   17836 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772276140.661865   17836 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772276140.661869   17836 computa

In [22]:
# XLM-RoBERTa-Large done

# Train IndicBERT

In [23]:
import shutil
import os

TEAM = "CUET_Synthetica"
SEEDS = [3407]

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Training IndicBERT v2 - Seed {seed}_1")
    print(f"{'='*60}")
    
    !python train_updated.py \
      --train "train_clean.csv" \
      --test "TestV2_clean.csv" \
      --team "$TEAM" \
      --model "ai4bharat/IndicBERTv2-MLM-only" \
      --epochs 10 \
      --max_len 128 \
      --lr 2e-5 \
      --batch 32 \
      --val_size 0.2 \
      --seed {seed} \
      --early_stopping_patience 3
    
    if os.path.exists("submissions/CUET_Synthetica.csv"):
        shutil.move("submissions/CUET_Synthetica.csv", 
                    f"submissions/CUET_Synthetica_IndicBERT_{seed}_1.csv")
        shutil.move("submissions/CUET_Synthetica_probs.npy", 
                    f"submissions/CUET_Synthetica_IndicBERT_{seed}_1_probs.npy")
        print(f"\n✅ Seed {seed} DONE\n")
    else:
        print(f"\n❌ Seed {seed} FAILED\n")

print("\n" + "="*60)
print("🎉 ALL IndicBERT MODELS COMPLETE!")
print("="*60)


Training IndicBERT v2 - Seed 3407_1
2026-02-28 11:26:29.221851: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772277989.243805   22171 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772277989.250349   22171 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772277989.268051   22171 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772277989.268077   22171 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772277989.268080   22171 computation_

In [24]:
import shutil
import os

TEAM = "CUET_Synthetica"
SEEDS = [42]

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Training IndicBERT v2 - Seed {seed}_2")
    print(f"{'='*60}")
    
    !python train_updated.py \
      --train "train_clean.csv" \
      --test "TestV2_clean.csv" \
      --team "$TEAM" \
      --model "ai4bharat/IndicBERTv2-MLM-only" \
      --epochs 5 \
      --max_len 128 \
      --lr 3e-5 \
      --batch 8 \
      --val_size 0.2 \
      --seed {seed} \
      --early_stopping_patience 3
    
    if os.path.exists("submissions/CUET_Synthetica.csv"):
        shutil.move("submissions/CUET_Synthetica.csv", 
                    f"submissions/CUET_Synthetica_IndicBERT_{seed}_2.csv")
        shutil.move("submissions/CUET_Synthetica_probs.npy", 
                    f"submissions/CUET_Synthetica_IndicBERT_{seed}_2_probs.npy")
        print(f"\n✅ Seed {seed} DONE\n")
    else:
        print(f"\n❌ Seed {seed} FAILED\n")

print("\n" + "="*60)
print("🎉 ALL IndicBERT MODELS COMPLETE!")
print("="*60)


Training IndicBERT v2 - Seed 42_2
2026-02-28 11:33:16.084206: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772278396.105535   23332 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772278396.111995   23332 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772278396.129661   23332 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772278396.129709   23332 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772278396.129716   23332 computation_pl

In [ ]:
import shutil
import os

TEAM = "CUET_Synthetica"
SEEDS = [42]

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Training IndicBERT v2 - Seed {seed}_3")
    print(f"{'='*60}")
    
    !python train_updated.py \
      --train "train_clean.csv" \
      --test "TestV2_clean.csv" \
      --team "$TEAM" \
      --model "ai4bharat/IndicBERTv2-MLM-only" \
      --epochs 5 \
      --max_len 128 \
      --lr 2e-5 \
      --batch 16 \
      --val_size 0.2 \
      --seed {seed} \
      --early_stopping_patience 3
    
    if os.path.exists("submissions/CUET_Synthetica.csv"):
        shutil.move("submissions/CUET_Synthetica.csv", 
                    f"submissions/CUET_Synthetica_IndicBERT_{seed}_3.csv")
        shutil.move("submissions/CUET_Synthetica_probs.npy", 
                    f"submissions/CUET_Synthetica_IndicBERT_{seed}_3_probs.npy")
        print(f"\n✅ Seed {seed} DONE\n")
    else:
        print(f"\n❌ Seed {seed} FAILED\n")

print("\n" + "="*60)
print("🎉 ALL IndicBERT MODELS COMPLETE!")
print("="*60)


Training IndicBERT v2 - Seed 42_3
2026-02-28 11:39:30.875410: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772278770.898780   25615 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772278770.905317   25615 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772278770.922383   25615 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772278770.922416   25615 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772278770.922421   25615 computation_pl

In [26]:
import shutil
import os

TEAM = "CUET_Synthetica"
SEEDS = [42]

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Training IndicBERT v2 - Seed {seed}_4")
    print(f"{'='*60}")
    
    !python train_updated.py \
      --train "train_clean.csv" \
      --test "TestV2_clean.csv" \
      --team "$TEAM" \
      --model "ai4bharat/IndicBERTv2-MLM-only" \
      --epochs 5 \
      --max_len 128 \
      --lr 3e-5 \
      --batch 32 \
      --val_size 0.2 \
      --seed {seed} \
      --early_stopping_patience 3
    
    if os.path.exists("submissions/CUET_Synthetica.csv"):
        shutil.move("submissions/CUET_Synthetica.csv", 
                    f"submissions/CUET_Synthetica_IndicBERT_{seed}_4.csv")
        shutil.move("submissions/CUET_Synthetica_probs.npy", 
                    f"submissions/CUET_Synthetica_IndicBERT_{seed}_4_probs.npy")
        print(f"\n✅ Seed {seed} DONE\n")
    else:
        print(f"\n❌ Seed {seed} FAILED\n")

print("\n" + "="*60)
print("🎉 ALL IndicBERT MODELS COMPLETE!")
print("="*60)


 91%|███████████████████████████████████████    | 10/11 [00:02<00:00,  3.62it/s]
                                                                                
{'eval_loss': 0.46428102254867554, 'eval_macro_precision': 0.7679050226586103, 'eval_macro_recall': 0.7680112960199474, 'eval_macro_f1': 0.7679418326034868, 'eval_runtime': 3.1429, 'eval_samples_per_second': 207.133, 'eval_steps_per_second': 3.5, 'epoch': 2.0}
100%|███████████████████████████████████████████| 11/11 [00:02<00:00,  4.35it/s]
{'loss': 0.4733, 'grad_norm': 3.6849069595336914, 'learning_rate': 1.85514663069967e-05, 'epoch': 2.44}
 91%|███████████████████████████████████████    | 10/11 [00:02<00:00,  3.47it/s]
                                                                                
{'eval_loss': 0.43803247809410095, 'eval_macro_precision': 0.7727234114848793, 'eval_macro_recall': 0.7729037193751298, 'eval_macro_f1': 0.7726311616228361, 'eval_runtime': 3.2817, 'eval_samples_per_second': 198.37, 'eval_steps_p

In [27]:
# IndicBERT done

In [28]:
import os

print("Checking submissions folder:")
print("="*50)

expected_files = [
  "CUET_Synthetica_MuRIL_3407_1.csv",
  "CUET_Synthetica_MuRIL_3407_1_probs.npy",
  "CUET_Synthetica_MuRIL_3407_2.csv",
  "CUET_Synthetica_MuRIL_3407_2_probs.npy",
  "CUET_Synthetica_MuRIL_3407_3_.csv",
  "CUET_Synthetica_MuRIL_3407_3_probs.npy",
  "CUET_Synthetica_MuRIL_3407_4.csv",
  "CUET_Synthetica_MuRIL_3407_4_probs.npy",
  "CUET_Synthetica_XLMRobertaLarge_3407_1.csv",
  "CUET_Synthetica_XLMRobertaLarge_3407_1_probs.npy",
  "CUET_Synthetica_XLMRobertaLarge_3407_2.csv",
  "CUET_Synthetica_XLMRobertaLarge_3407_2_probs.npy",
  "CUET_Synthetica_XLMRobertaLarge_3407_3.csv",
  "CUET_Synthetica_XLMRobertaLarge_3407_3_probs.npy",
  "CUET_Synthetica_XLMRobertaLarge_3407_4.csv",
  "CUET_Synthetica_XLMRobertaLarge_3407_4_probs.npy",
  "CUET_Synthetica_IndicBERT_3407_1.csv",
  "CUET_Synthetica_IndicBERT_3407_1_probs.npy",
  "CUET_Synthetica_IndicBERT_42_2.csv",
  "CUET_Synthetica_IndicBERT_42_2_probs.npy",
  "CUET_Synthetica_IndicBERT_42_3.csv",
  "CUET_Synthetica_IndicBERT_42_3_probs.npy",
  "CUET_Synthetica_IndicBERT_42_4.csv",
  "CUET_Synthetica_IndicBERT_42_4_probs.npy"

]

all_found = True
for f in expected_files:
    path = f"submissions/{f}"
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"✓ {f:50s} {size:>10,} bytes")
    else:
        print(f"✗ MISSING: {f}")
        all_found = False

print("="*50)
if all_found:
    print("✓ ALL 12 MODELS SAVED CORRECTLY!")
else:
    print("⚠️ Some files missing - re-run the missing models")

Checking submissions folder:
✓ CUET_Synthetica_MuRIL_3407_1.csv                      263,433 bytes
✓ CUET_Synthetica_MuRIL_3407_1_probs.npy                  7,432 bytes
✓ CUET_Synthetica_MuRIL_3407_2.csv                      263,433 bytes
✓ CUET_Synthetica_MuRIL_3407_2_probs.npy                  7,432 bytes
✓ CUET_Synthetica_MuRIL_3407_3_.csv                     263,469 bytes
✓ CUET_Synthetica_MuRIL_3407_3_probs.npy                  7,432 bytes
✓ CUET_Synthetica_MuRIL_3407_4.csv                      263,469 bytes
✓ CUET_Synthetica_MuRIL_3407_4_probs.npy                  7,432 bytes
✓ CUET_Synthetica_XLMRobertaLarge_3407_1.csv            263,485 bytes
✓ CUET_Synthetica_XLMRobertaLarge_3407_1_probs.npy        7,432 bytes
✓ CUET_Synthetica_XLMRobertaLarge_3407_2.csv            263,417 bytes
✓ CUET_Synthetica_XLMRobertaLarge_3407_2_probs.npy        7,432 bytes
✓ CUET_Synthetica_XLMRobertaLarge_3407_3.csv            263,433 bytes
✓ CUET_Synthetica_XLMRobertaLarge_3407_3_probs.npy        7,4

# Create Ensemble of 12 model

In [29]:
import numpy as np
import pandas as pd
import os

TEAM = "CUET_Synthetica"
SUB_DIR = "submissions"

print("\n" + "="*70)
print("CREATING FINAL ENSEMBLE (STRICT 12-MODEL SELECTION)")
print("="*70)

# Define the EXACT 12 models to use

target_files = [
    # IndicBERT (4)
    "CUET_Synthetica_IndicBERT_3407_1_probs.npy",
    "CUET_Synthetica_IndicBERT_42_2_probs.npy",
    "CUET_Synthetica_IndicBERT_42_3_probs.npy",
    "CUET_Synthetica_IndicBERT_42_4_probs.npy",

    # MuRIL (4)
    "CUET_Synthetica_MuRIL_3407_1_probs.npy",
    "CUET_Synthetica_MuRIL_3407_2_probs.npy",
    "CUET_Synthetica_MuRIL_3407_3_probs.npy",
    "CUET_Synthetica_MuRIL_3407_4_probs.npy",

    # XLM-R Large (4)
    "CUET_Synthetica_XLMRobertaLarge_3407_1_probs.npy",
    "CUET_Synthetica_XLMRobertaLarge_3407_2_probs.npy",
    "CUET_Synthetica_XLMRobertaLarge_3407_3_probs.npy",
    "CUET_Synthetica_XLMRobertaLarge_3407_4_probs.npy",
]

prob_files = []
missing_files = []

# Verify existence of each specific file
for f in target_files:
    path = os.path.join(SUB_DIR, f)
    if os.path.exists(path):
        prob_files.append(f)
    else:
        missing_files.append(f)

# Error handling
if missing_files:
    print("\n❌ ERROR: The following expected files were NOT found in /submissions:")
    for m in missing_files:
        print(f"   - {m}")
    raise FileNotFoundError("Missing probability files required for ensemble.")

if len(prob_files) != 12:
    raise RuntimeError(f"Expected 12 models, but verified {len(prob_files)}.")

print("\nSuccessfully verified 12 models for ensembling.")

# Load probabilities

all_probs = []
model_names = []

for pf in prob_files:
    path = os.path.join(SUB_DIR, pf)
    probs = np.load(path)
    all_probs.append(probs)
    
    model_name = pf.replace("CUET_Synthetica_", "").replace("_probs.npy", "")
    model_names.append(model_name)
    print(f"✅ Loaded: {model_name:40s} | Shape: {probs.shape}")


# Simple Soft-Voting Ensemble (Averaging)

print("\n" + "="*70)
print("COMPUTING MEAN PROBABILITIES")
print("="*70)

# stacked shape: (12, Number_of_Samples, 2)
stacked = np.stack(all_probs, axis=0)  
avg_probs = stacked.mean(axis=0)

# Get final predicted class (0 or 1)
pred_ids = np.argmax(avg_probs, axis=1)

label_map = {0: "Non-Abusive", 1: "Abusive"}
pred_labels = [label_map[int(i)] for i in pred_ids]

# Save submission

# Ensure TestV2_clean.csv is in the same directory
test_df = pd.read_csv("TestV2_clean.csv")

# We keep the original columns from the test set and add our predictions
submission_df = test_df.copy()
submission_df["Class"] = pred_labels

output_file = f"{SUB_DIR}/{TEAM}_FINAL_ENSEMBLE_12models.csv"
submission_df.to_csv(output_file, index=False, encoding="utf-8")

print(f"\n📁 Final Ensemble Saved: {output_file}")

# Quick Analysis

print("\n" + "="*70)
print("PREDICTION STATS:")
print("="*70)
print(submission_df["Class"].value_counts(normalize=True) * 100)
print(f"\nAverage Ensemble Confidence: {avg_probs.max(axis=1).mean()*100:.2f}%")
print("="*70 + "\n")


CREATING FINAL ENSEMBLE (STRICT 12-MODEL SELECTION)

Successfully verified 12 models for ensembling.
✅ Loaded: IndicBERT_3407_1                         | Shape: (913, 2)
✅ Loaded: IndicBERT_42_2                           | Shape: (913, 2)
✅ Loaded: IndicBERT_42_3                           | Shape: (913, 2)
✅ Loaded: IndicBERT_42_4                           | Shape: (913, 2)
✅ Loaded: MuRIL_3407_1                             | Shape: (913, 2)
✅ Loaded: MuRIL_3407_2                             | Shape: (913, 2)
✅ Loaded: MuRIL_3407_3                             | Shape: (913, 2)
✅ Loaded: MuRIL_3407_4                             | Shape: (913, 2)
✅ Loaded: XLMRobertaLarge_3407_1                   | Shape: (913, 2)
✅ Loaded: XLMRobertaLarge_3407_2                   | Shape: (913, 2)
✅ Loaded: XLMRobertaLarge_3407_3                   | Shape: (913, 2)
✅ Loaded: XLMRobertaLarge_3407_4                   | Shape: (913, 2)

COMPUTING MEAN PROBABILITIES

📁 Final Ensemble Saved: submissions/CUE

# Download Zip File

In [30]:
import shutil
import os
from IPython.display import FileLink

# Clean up
broken_file = "submissions/CUET_Synthetica_MuRIL_3407_4_1.csv"
if os.path.exists(broken_file):
    os.remove(broken_file)
    print(f"🗑️ Removed outlier: {broken_file}")

# Create the ZIP archive
shutil.make_archive('CUET_Final_Submission', 'zip', 'submissions')

print(" ZIP created successfully.")
FileLink('CUET_Final_Submission.zip')

 ZIP created successfully.


/kaggle/working/CUET_Final_Submission.zip

In [31]:
# Main code done

# Detailed Dataset Statistics

In [32]:
# TABLE 1: Detailed Dataset Statistics

def calculate_dataset_statistics(df, text_col='Text', label_col='Class'):

    statistics = {}
    
    for class_label in df[label_col].unique():
        class_data = df[df[label_col] == class_label]
        
        # Split text into words
        all_words = []
        lengths = []
        
        for text in class_data[text_col]:
            words = str(text).split()
            all_words.extend(words)
            lengths.append(len(words))
        
        statistics[class_label] = {
            'Total words': len(all_words),
            'Unique words': len(set(all_words)),
            'Max len (words)': max(lengths) if lengths else 0,
            'Avg word per text': round(np.mean(lengths), 1) if lengths else 0,
            'Min len (words)': min(lengths) if lengths else 0,
            'Median len (words)': round(np.median(lengths), 1) if lengths else 0
        }
    
    return statistics

train_df = pd.read_csv('train_clean.csv')

# Calculate statistics
stats = calculate_dataset_statistics(train_df)

# Create DataFrame for display
stats_df = pd.DataFrame(stats).T

print("\n" + "="*100)
print("TABLE 1: Detailed Statistics of Training Dataset")
print("="*100)
print(stats_df.to_string())
print("="*100)

# Save as CSV
stats_df.to_csv('paper_outputs/tables/table1_dataset_statistics.csv')

# Generate LaTeX table
latex_table = stats_df.to_latex(
    caption="Detailed statistics of each class in the training set",
    label="tab:dataset_stats",
    column_format='l' + 'r'*len(stats_df.columns)
)

with open('paper_outputs/latex/table1_dataset_statistics.tex', 'w') as f:
    f.write(latex_table)

print("\n✓ Table 1 saved to paper_outputs/")


TABLE 1: Detailed Statistics of Training Dataset
             Total words  Unique words  Max len (words)  Avg word per text  Min len (words)  Median len (words)
Non-Abusive      22631.0        9612.0            257.0               13.6              4.0                10.0
Abusive          24285.0       10492.0            381.0               15.3              3.0                11.0

✓ Table 1 saved to paper_outputs/
